# Loading Qwen3 0.6B model

In [1]:
# Install KerasHub
!pip install -q git+https://github.com/keras-team/keras-hub.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 7.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
keras-nlp 0.18.1 requires keras-hub==0.18.1, but you have keras-hub 0.24.0.dev0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
pydrive2 1.21.3 requires cryptography<44

In [2]:
# Import Libraries
import keras_hub
import numpy as np
import keras
from dataclasses import dataclass
from typing import Optional, List, Dict, Any

2025-12-10 07:00:47.122276: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765350047.409637      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765350047.496039      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
class LLMConfig:
    def __init__(self):
        self.max_tokens = 50
        self.temperature = 0.7
        self.top_p = 1.0

class LLMClient:
    def __init__(self, config: Optional[LLMConfig] = None):
        self.config = config or LLMConfig()
        self.model = None
        self._is_initialized = False
    
    def initialize(self): # Renommé de initialize1 à initialize pour la cohérence
        from typing import Optional
        import keras_hub
        import keras
        """Initialise le modèle LLM"""
        # Les imports transformers, numpy et keras ne sont pas nécessaires pour l'utilisation basique de keras_hub.models.Qwen3CausalLM
        print("\nLoading Qwen3 0.6B model...")
        # Le modèle chargé gère la tokenisation en interne par défaut
        self.model = keras_hub.models.Qwen3CausalLM.from_preset("kaggle://keras/qwen-3/keras/qwen3_0.6b_en/1")
        # Compiler le modèle pour définir les paramètres d'échantillonnage
        self.model.compile(
                sampler="greedy" 
        )
        self._is_initialized = True
        print("Le modèle est prêt !")

    def generate(self, prompt: str) -> str:
        """Génère du texte à partir du prompt"""
        if not self._is_initialized:
            raise RuntimeError("Le modèle n'a pas été initialisé. Appelez la méthode initialize() d'abord.")

        print(f"Prompt envoyé au modèle : {prompt}")
        # La méthode generate() de keras_hub.models.CausalLM accepte directement une chaîne de caractères (ou une liste de chaînes)
        # et gère automatiquement le pré-traitement (tokenization) et le post-traitement (détokénisation).
        output = self.model.generate(
            prompt,
            max_length=self.config.max_tokens,
        )
        # La sortie est déjà une chaîne de caractères (ou une liste de chaînes)
        return output[0] if isinstance(output, list) else output

In [4]:
client = LLMClient()

In [5]:
client.initialize()


Loading Qwen3 0.6B model...


2025-12-10 07:01:10.950922: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Le modèle est prêt !


In [6]:
prompt = "Explain how the Qwen-3 model works in a few sentences."
generated_text = client.generate(prompt)

Prompt envoyé au modèle : Explain how the Qwen-3 model works in a few sentences.
generated token ids =  Tensor("strided_slice_28:0", shape=(50,), dtype=int32)


I0000 00:00:1765350121.315681      86 service.cc:148] XLA service 0x7d47b0007cb0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1765350121.316619      86 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1765350121.385530      86 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [7]:
print(generated_text)

Explain how the Qwen-3 model works in a few sentences. Qwen-3 is a large language model developed by Alibaba Group, which is trained on a massive amount of data. It has a large vocabulary and can understand and generate text


# Using transformers

In [8]:
class LLMConfig:
    """Configuration pour le client LLM"""
    model_name: str =  "Qwen/Qwen3-0.6B"     # "Qwen/Qwen2.5-3B-Instruct" 
    model_path: Optional[str] = None
    max_tokens: int = 2048
    temperature: float = 0.7
    top_p: float = 0.9
    device: str = "cpu"  # Choisissez 'cuda' si vous utilisez un GPU compatible

# Définition de la classe LLMClient
class LLMClient:
    def __init__(self, config: Optional[LLMConfig] = None):
        self.config = config or LLMConfig()
        self.model = None
        self.tokenizer = None
        self._is_initialized = False
    
    def initialize(self):
        """Initialise le modèle LLM"""
        from transformers import AutoModelForCausalLM, AutoTokenizer

        print("Chargement du modèle...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.config.model_name,
            device_map="auto" if self.config.device == "auto" else None,
        )
        self._is_initialized = True
        print("Le modèle est prêt !")

    def generate(self, prompt: str) -> str:
        """Génère du texte à partir du prompt"""
        if not self._is_initialized:
            raise RuntimeError("Le modèle n'a pas été initialisé. Appelez la méthode initialize() d'abord.")

        print(f"Prompt envoyé au modèle : {prompt}")
        inputs = self.tokenizer.encode(prompt, return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(
            inputs,
            max_length=self.config.max_tokens,
            temperature=self.config.temperature,
            top_p=self.config.top_p
        )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

In [9]:
config = LLMConfig()
print(config)

In [10]:
client = LLMClient(config)

In [11]:
client.initialize()

Chargement du modèle...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Le modèle est prêt !


In [12]:
# Exemple d'utilisation : générer du texte à partir d'un prompt
prompt = "Explain how the Qwen-3 model works in a few sentences"
response = client.generate(prompt)
    
print("\nRéponse générée par le LLM :")
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Prompt envoyé au modèle : Explain how the Qwen-3 model works in a few sentences

Réponse générée par le LLM :
Explain how the Qwen-3 model works in a few sentences, focusing on its architecture, training process, and performance.
Qwen-3 is a large-scale language model developed by Alibaba Group. It is the first to achieve the capability of self-attention with a larger scale, which is the first to have the capability of multi-scale attention in a language model. The model is designed to solve problems in the field of AI and the development of the field of AI and related fields. The model is also capable of being used for the development of the field of education and the field of health care and other areas.
Qwen-3 is a large-scale language model with a very large vocabulary and a very large context size. It is developed using the BERT architecture and has been trained on a large dataset. It has been widely used in the field of AI and related fields, and it is also capable of being used 